### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [2]:
from unsloth import FastModel
from unsloth import FastLanguageModel

import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B",
    max_seq_length = 2048,
    load_in_4bit = True,
)



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update a small amount of parameters!

In [3]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

<a name="Data"></a>
### Data Prep
Konuşma tarzı fine-tune için **Llama-3.1** chat template'ini kullanıyoruz (Llama-3.2 ile uyumlu). Llama çok turlu konuşmaları şöyle render eder:

```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Hello!<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Hey there!<|eot_id|>
```

`get_chat_template` fonksiyonu doğru template'i verir. `zephyr, chatml, mistral, llama, llama-3, llama-3.1, phi3, phi4, qwen2.5, gemma3` ve daha fazlasını destekler. **Bu model Llama olduğu için `llama-3.1` seçiyoruz.**

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",  # Llama 3.2, llama-3.1 template'ini kullanır (Gemma DEĞİL)
)

In [10]:
from datasets import load_dataset

dataset = load_dataset(
    "gorkemergune/ayarlicazhocam_finetune",
    split="train"
)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.56M [00:00<?, ?B/s]

data/en-00000-of-00001.parquet:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2009 [00:00<?, ? examples/s]

Generating tr split:   0%|          | 0/1004 [00:00<?, ? examples/s]

Generating en split:   0%|          | 0/1005 [00:00<?, ? examples/s]

We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

In [11]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

Let's see how row 100 looks like!

In [12]:
dataset[100]

{'messages': [{'content': 'Python decorator ne işe yarar?',
   'images': None,
   'role': 'user',
   'thinking': None,
   'tool_calls': None},
  {'content': "Decorator, bir fonksiyonun davranışını değiştirmek için kullanılan bir Python özelliği. @ sembolüyle kullanılır. Mesela @login_required bir fonksiyonun sadece giriş yapmış kullanıcılar tarafından çağrılmasını sağlar. @staticmethod, @classmethod, @property yerleşik decorator'lar. Kendi decorator'ını yazabilirsin — loglama, zamanlama, caching için çok kullanışlı. Flask ve Django'da route tanımlamak için decorator kullanılır: @app.route('/home'). Decorator aslında bir fonksiyon alıp yeni bir fonksiyon döndüren fonksiyon — higher-order function kavramı. Python'ın en elegant özelliklerinden biri.",
   'images': None,
   'role': 'assistant',
   'thinking': 'Python decorator, @, login_required, Flask, higher-order function.',
   'tool_calls': None}]}

We now have to apply the chat template for `Gemma-3` onto the conversations, and save it to `text`. We remove the `<bos>` token using removeprefix(`'<bos>'`) since we're finetuning. The Processor will add this token before training and the model expects only one.

In [13]:
dataset[0]

{'messages': [{'content': 'What is the banking management system project?',
   'images': None,
   'role': 'user',
   'thinking': None,
   'tool_calls': None},
  {'content': "It's a banking management system Gorkem wrote in Python. It probably simulates basic banking operations — account creation, deposits, withdrawals, transfers, balance inquiries. These kinds of projects are great for beginners because you practice OOP, learn data structures, handle file operations. Every developer should have a project like this in their portfolio when starting out.",
   'images': None,
   'role': 'assistant',
   'thinking': 'Banking system, OOP practice, beginner level project.',
   'tool_calls': None}]}

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix(tokenizer.bos_token)  # literal '<bos>' yerine gerçek BOS (Llama: <|begin_of_text|>)
        for convo in convos
    ]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Let's see how the chat template did! Notice there is no `<bos>` token as the processor tokenizer will be adding one.

In [16]:
dataset[50]["text"]

"<|begin_of_text|><start_of_turn>user\nwhat are design patterns every developer should know?<end_of_turn>\n<start_of_turn>model\nDesign patterns are reusable solutions to common problems. Creational: Singleton (one instance — database connection, but often an anti-pattern, use dependency injection instead), Factory (create objects without specifying exact class), Builder (construct complex objects step by step — useful for objects with many optional parameters). Structural: Adapter (make incompatible interfaces work together), Decorator (add behavior dynamically — Python decorators), Facade (simple interface for complex subsystem), Proxy (control access to an object — lazy loading, caching, access control). Behavioral: Observer (pub/sub — event systems, React state), Strategy (swap algorithms at runtime), Command (encapsulate actions — undo/redo), Iterator (traverse collections without exposing internals). Don't force patterns — use them when the problem naturally fits. The Gang of Fou

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,   # persona/kimlik öğretmek için tekrar gerekir
        max_steps=-1,         # epoch'u kullan (max_steps devre dışı)
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",

        fp16=True,   # Tesla T4'te bf16 yok, fp16 doğru
        bf16=False,
    ),
)

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes! Unsloth now auto-detects the instruction and response parts from the tokenizer's chat template, so we don't need to pass `instruction_part` and `response_part` anymore. You can still pass them explicitly if you use a custom chat template.

In [18]:
print(type(trainer.data_collator))
print(type(model))
print(type(tokenizer))

<class 'trl.trainer.sft_trainer.DataCollatorForLanguageModeling'>
<class 'peft.peft_model.PeftModelForCausalLM'>
<class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>


In [19]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)

Unsloth: Auto-detected instruction_part = '<end_of_turn>\n<start_of_turn>user\n' and response_part = '<end_of_turn>\n<start_of_turn>model\n'


Map:   0%|          | 0/2009 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2009 [00:00<?, ? examples/s]

Unsloth: Removed 1872 out of 2009 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Let's verify masking the instruction part is done! Let's print the 100th row again.  Notice how the sample only has a single `<bos>` as expected!

In [20]:
tokenizer.decode(trainer.train_dataset[30]["input_ids"])

"<|begin_of_text|><|begin_of_text|><start_of_turn>user\nmerhaba nasılsın<end_of_turn>\n<start_of_turn>model\nMerhaba! Ben bir yapay zeka olduğum için tam olarak 'nasılım' diyemem ama hazırım ve enerjiyim yerinde diyelim. Sen nasılsın, bugün sana nasıl yardımcı olabilirim?<end_of_turn>\n"

Now let's print the masked out example - you should see only the answer is present:

In [21]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[30]["labels"]]).replace(tokenizer.pad_token, " ")

"                            Merhaba! Ben bir yapay zeka olduğum için tam olarak 'nasılım' diyemem ama hazırım ve enerjiyim yerinde diyelim. Sen nasılsın, bugün sana nasıl yardımcı olabilirim?<end_of_turn>\n"

In [22]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
3.051 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [23]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 137 | Num Epochs = 2 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,156,928 of 3,224,906,752 (0.38% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.861300
2,2.408500
3,2.626500
4,2.484700
5,2.393500
6,2.455000
7,2.466300
8,2.395500
9,2.305900
10,2.464600


In [24]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

117.5247 seconds used for training.
1.96 minutes used for training.
Peak reserved memory = 3.051 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 20.95 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Modeli Unsloth native inference ile çalıştıralım! Llama-3.2 için makul ayarlar: `temperature = 0.7, top_p = 0.9`. (Gemma'ya özgü `top_k = 64` ayarını KULLANMIYORUZ.)

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)  # inference modunu aç (2x hızlı)

messages = [
    {"role": "user", "content": "Continue the sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,  # generation için şart
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 64,  # daha uzun çıktı için artır
    # Llama ayarları (Gemma'nın top_k=64'ü değil)
    temperature = 0.7, top_p = 0.9,
)
tokenizer.batch_decode(outputs)

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
messages = [
    {"role": "user", "content": "Why is the sky blue?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,  # generation için şart
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 64,  # daha uzun çıktı için artır
    # Llama ayarları
    temperature = 0.7, top_p = 0.9,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [27]:
model.save_pretrained("Llama_3_lora")  # Local saving
tokenizer.save_pretrained("Llama_3_lora")
# model.push_to_hub("HF_ACCOUNT/gemma_3_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma_3_lora", token = "YOUR_HF_TOKEN") # Online saving

('Llama_3_lora/tokenizer_config.json',
 'Llama_3_lora/special_tokens_map.json',
 'Llama_3_lora/chat_template.jinja',
 'Llama_3_lora/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "Llama_3_lora",  # cell yukarısında kaydedilen dizinle AYNI olmalı
        max_seq_length = 2048,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Who is Gorkem?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,  # generation için şart
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 256,  # persona cevabının tamamını görmek için artırıldı
    # Llama ayarları
    temperature = 0.7, top_p = 0.9,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)